In [1]:
#!pip install delta-spark==4.0.0

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

# Create Spark session
spark = SparkSession.builder \
    .master("spark://spark-master:7077")  \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0,io.delta:delta-spark_2.13:4.0.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .appName("CoinbaseKafkaConsumer") \
    .getOrCreate()



In [4]:
from pyspark.sql.functions import col, window, avg

# Cast price to double for aggregation

spark.sparkContext.setLogLevel("WARN")

# Define schema for Coinbase ticker messages
schema = StructType([
    StructField("type", StringType()),
    StructField("sequence", LongType()),
    StructField("product_id", StringType()),
    StructField("price", StringType()),
    StructField("open_24h", StringType()),
    StructField("volume_24h", StringType()),
    StructField("low_24h", StringType()),
    StructField("high_24h", StringType()),
    StructField("volume_30d", StringType()),
    StructField("best_bid", StringType()),
    StructField("best_bid_size", StringType()),
    StructField("best_ask", StringType()),
    StructField("best_ask_size", StringType()),
    StructField("side", StringType()),
    StructField("time", TimestampType()),
    StructField("trade_id", LongType()),
    StructField("last_size", StringType())
])

# Kafka broker and topic
KAFKA_BROKER = "broker-1:19092,broker-2:19092,broker-3:19092"   # adjust if docker-compose uses "broker:9092"
KAFKA_TOPIC = "coinbase_feed"

# Read from Kafka
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BROKER) \
    .option("subscribe", KAFKA_TOPIC) \
    .option("startingOffsets", "latest") \
    .load()

# Kafka value is in binary, so we need to cast to string
json_df = df.selectExpr("CAST(value AS STRING) as json_value")

# Parse JSON using schema
parsed_df = json_df.select(from_json(col("json_value"), schema).alias("data")).select("data.*")
parsed_df = parsed_df.withColumn("price", col("price").cast("double"))
windowed_avg_df = parsed_df \
    .withWatermark("time", "1 minutes") \
    .groupBy(
        window(col("time"), "1 minutes"),
        col("product_id")
    ) \
    .agg(
        avg("price").alias("avg_price")
    )



In [ ]:
# Write to console in append mode
query = windowed_avg_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "./delta/events/_checkpoints/") \
    .toTable("events")

query.awaitTermination()

25/08/30 19:38:29 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
25/08/30 19:38:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 907:===============>                                      (58 + 2) / 200]